Run with shape: (1027368, 18)  
gva1 L1.gva1 total_assets employees tfp_wav1 | gmm(gva1, 2:4) gmm(total_assets, 2:3) iv(tfp_wav1) | timedumm  
- With 'collapse': takes just over 1 minute to run

In [1]:
# Install dependencies if needed:
# !pip install "numpy<2.0.0" "pandas<2.2.0" pydynpd

import pandas as pd
import numpy as np
from pydynpd import regression
from typing import Any

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

if np.__version__ >= "2.0.0":
    raise ImportError("NumPy version must be less than 2.0.0")
if pd.__version__ >= "2.2.0":
    raise ImportError("Pandas version must be less than 2.2.0")
print("Libraries loaded successfully.")

NumPy version: 1.26.4
Pandas version: 2.1.4
Libraries loaded successfully.


In [2]:
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")

table_panel_name = "working_yearly_n"
parquet_path = dirs.tmp_dir / f"{table_panel_name}.parquet"
print(f"Parquet file path: {parquet_path}")
df_panel = pd.read_parquet(parquet_path)
if df_panel is None:
    print(f"Failed to load DataFrame from {parquet_path}")
    df_panel = pd.read_csv(dirs.tmp_dir / f"{table_panel_name}.csv")
if df_panel is None:
    raise FileNotFoundError(f"Could not load DataFrame from {parquet_path} or CSV.")
# log_vars = ["gva1", "total_assets", "employees"]
df_panel_dyn = (
    df_panel
    .dropna(subset=["y", "k", "l"])
    .drop_duplicates(subset=["registered_number", "year"], keep="last")
    # .assign(**{
    #     f'ln_{p}': df_panel[p].apply(lambda x: np.log(x) if x > 0 else None) for p in log_vars
    # })
)
print(f"DataFrame loaded successfully with shape: {df_panel_dyn.shape}")

def is_number(s: Any) -> bool:
    try:
        float(s)
        return True
    except (ValueError, TypeError):
        return False

def unpack_x_var(var_tuple: tuple[str, ...]) -> str:
    var, *lags = var_tuple
    if len(lags) == 0:
        return var
    elif len(lags) == 1:
        if lags[0] == 0:
            return var
        else:
            return f"L{lags[0]}.{var}"
    elif len(lags) == 2:
        if lags[0] == lags[1]:
            return f"L{lags[0]}.{var}"
        else:
            return f"L({lags[0]}:{lags[1]}).{var}"
    else:
        raise ValueError(f"Too many lags specified for variable {var}: {lags}")

def unpack_z_var(var_tuple: tuple[str, ...]) -> str:
    var, *lags = var_tuple
    if len(lags) == 0:
        return f"iv({var})"
    elif len(lags) == 1:
        if lags[0] == 0:
            return f"iv({var})"
        else:
            return f"iv(L{lags[0]}.{var})"
    elif len(lags) == 2:
        if lags[0] == lags[1]:
            if lags[0] == 0:
                return f"iv({var})"
            else:
                return f"iv(L{lags[0]}.{var})"
        elif is_number(lags[0]) and is_number(lags[1]) and int(lags[0]) > int(lags[1]):
            raise ValueError(f"Invalid lag range for variable {var}: {lags}")
        else:
            return f"gmm({var}, {lags[0]}:{lags[1]})"
    else:
        raise ValueError(f"Too many lags specified for variable {var}: {lags}")

Parquet file path: C:\Users\lazyst\Files\ucl\Dissertation\model\tmp\working_yearly_n.parquet
DataFrame loaded successfully with shape: (1045164, 49)


# PYDYNPD package

### Part 1: dependent-independent
- Basic: `n L1.n L2.n w k`. Don't distinguish between endogenous and predetermined variables here.
- `n`: my `ln(y)`, dependent variable

### Part 2: instrument creation
- `gmm(2:5)`. Use 2: if the variable is endogenous, 1: if predetermined.
- `.`: dot is no restriction on maximum lags.
- `?`: automatic mode: find maximum lags.
- `endo(list of variables)` is equivalent to `gmm(list of variables, 2:.)`
- `pred(list of variables)` is equivalent to `gmm(list of variables, 1:.)`
- `iv(k)`: strictly exogenous variable.

### Test results:
- `AR(1) test reject null`: first-order autocorrelation. Cannot use 1st lag of DV as instrument.
- `AR(2) test accept null`: no 2nd-order autocorrelation. Fine to use 2nd lag of DV as instrument, don't use as a regressor.
- `Hansen J reject`: instruments are not exogenous (overidentified)
  
### Ex: command strings
- `ln_gva1 L(1:1).ln_gva1 ln_total_assets ln_employees tfp_wav1 | gmm(ln_gva1, 2:5) gmm(ln_total_assets, 2:4) iv(tfp_wav1) | timedumm collapse`
--------------------------------------------------

In [3]:
import pandas as pd
from pydynpd.model_summary import model_summary
from pydynpd.regression import abond
from pydynpd.dynamic_panel_model import dynamic_panel_model

class DynResults:
    nobs: int
    cov: pd.DataFrame
    params: pd.Series
    table: pd.DataFrame
    model: dynamic_panel_model

    def __init__(self, nobs: int, cov: pd.DataFrame, params: pd.Series, table: pd.DataFrame, model: dynamic_panel_model):
        self.nobs = nobs
        self.cov = cov
        self.params = params
        self.table = table
        self.model = model

def get_dyn_results(mydpyd: abond) -> DynResults:
    m = mydpyd.models[0]
    variables = m.regression_table.variable.copy()
    vcov_arr = m.step_results[1].vcov
    vcov = pd.DataFrame(vcov_arr, columns=variables, index=variables)
    vcov.loc['wd1_y', 'wd1_y']
    regression_table = m.regression_table.copy()
    regression_table.set_index('variable', inplace=True)
    params = regression_table['coefficient']

    res = DynResults(
        nobs=m.num_obs,
        cov=vcov,
        params=params,
        table = regression_table,
        model = m
    )
    return res

In [4]:
import json
from f_2_extract import extract_structural, ModelSpec

mod_obj = {
    'Y': '',
    'X': [],
    'struct_map': {
        'k': 'beta_1',
        'l': 'beta_2',
        'wd1_y': 'delta_0',
        'L1.wd1_y': 'delta_1',
        'L2.wd1_y': 'delta_2',
        'L3.wd1_y': 'delta_3',
        'L4.wd1_y': 'delta_4',
        'L5.wd1_y': 'delta_5',
        'L1.y': 'phi_1',
        'L2.y': 'phi_2',
        'L3.y': 'phi_3',
        'L4.y': 'phi_4',
        'L5.y': 'phi_5'
    },
    'struct_calc': {
        'xi_0':     { 'name_wxs': ['wd1_k'],    'name_beta': 'k' },
        'zeta_0':   { 'name_wxs': ['wd1_l'],    'name_beta': 'l' },
        'xi_1':     { 'name_wxs': ['L1.wg1_k'], 'name_beta': 'k' },
        'zeta_1':   { 'name_wxs': ['L1.wg1_l'], 'name_beta': 'l' },
        'xi_2':     { 'name_wxs': ['L2.wg2_k'], 'name_beta': 'k' },
        'zeta_2':   { 'name_wxs': ['L2.wg2_l'], 'name_beta': 'l' },
        'xi_3':     { 'name_wxs': ['L3.wg3_k'], 'name_beta': 'k' },
        'zeta_3':   { 'name_wxs': ['L3.wg3_l'], 'name_beta': 'l' },
        'xi_4':     { 'name_wxs': ['L4.wg4_k'], 'name_beta': 'k' },
        'zeta_4':   { 'name_wxs': ['L4.wg4_l'], 'name_beta': 'l' },
        'xi_5':     { 'name_wxs': ['L5.wg5_k'], 'name_beta': 'k' },
        'zeta_5':   { 'name_wxs': ['L5.wg5_l'], 'name_beta': 'l' }
    }
}

def get_output_str(res: DynResults, instruments: list[tuple[str, int, int]], m_name: str) -> str:

    # Handle structural
    mod = ModelSpec(**mod_obj)
    struct = extract_structural(res, mod)      # type: ignore
    struct_for_df = {k: (*v[0:4], None) for k, v in struct.items() if k not in res.table.index}
    df_struct = pd.DataFrame.from_dict(struct_for_df, orient='index', columns=res.table.columns)
    # Add the structural rows to the dataframe
    res.table = pd.concat([res.table, df_struct])

    # Handle output formatting
    model = res.model
    s = model_summary()
    if model.options.steps == 2:
        str_steps = 'two-step '
    elif model.options.steps == 1:
        str_steps = 'one-step '
    else:
        str_steps = str(model.options.steps) + '-step '      # type: ignore
    if model.options.level:
        str_gmm = 'system GMM'
    else:
        str_gmm = 'difference GMM'

    # Get struct_calc ready
    calc_list = []
    if mod.struct_calc is not None:
        if isinstance(mod.struct_calc, list):
            calc_list = mod.struct_calc
        elif isinstance(mod.struct_calc, dict):
            calc_list = list(mod.struct_calc.keys())

    instr_obj = [{ 'name': ins[0], 'from': ins[1], 'to': ins[2] } for ins in instruments]

    # Get regression table ready
    df_rtable = res.table.copy()
    df_rtable.reset_index(inplace=True)
    df_rtable.rename(columns={'index': 'y'}, inplace=True)

    to_print = []
    # to_print.append(model.command_str)
    to_print.append(' Dynamic panel-data estimation, ' + str_steps + str_gmm + ". Model '" + m_name + "':")
    to_print.append(s.basic_information(model))
    # Print the structural map and calculations (keys)
    to_print.append(f"struct_map: {json.dumps(mod.struct_map)}")
    to_print.append(f"struct_calc: {json.dumps(calc_list)}")
    to_print.append(f"instruments: {json.dumps(instr_obj)}")
    to_print.append('Generated Command String:')
    to_print.append(model.command_str)
    to_print.append(df_rtable.to_markdown(index=False))
    to_print.append(s.test_results(model))
    to_print.append(('=' * 80) + '\n')
    return "\n".join(to_print)

ModuleNotFoundError: No module named 'rpds.rpds'

In [ ]:
models = {
    # Different instrument lags
    "k-pred": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 3),
            ("k", 1, 2),
            ("l", 2, 3),
            ("wd1_y", 2, 3),
            ("wd1_k", 1, 2),
            ("wd1_l", 2, 3)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'factor',
        'include': False
    },
    "l-pred": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 3),
            ("k", 2, 3),
            ("l", 1, 2),
            ("wd1_y", 2, 3),
            ("wd1_k", 1, 2),
            ("wd1_l", 2, 3)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'factor'
    },
    "y-pred": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 1, 2),
            ("k", 2, 3),
            ("l", 2, 3),
            ("wd1_y", 2, 3),
            ("wd1_k", 1, 2),
            ("wd1_l", 2, 3)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'instruments',
        'include': False
    },
    "both-endog": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 3),
            ("k", 2, 3),
            ("l", 2, 3),
            ("wd1_y", 2, 3),
            ("wd1_k", 1, 2),
            ("wd1_l", 2, 3)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'factor',
        'include': False
    },
    "deep-4": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 4),
            ("k", 2, 4),
            ("l", 2, 4),
            ("wd1_y", 2, 4),
            ("wd1_k", 2, 4),
            ("wd1_l", 2, 4)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'instruments'
    },
    "deep-5": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 5),
            ("k", 2, 5),
            ("l", 2, 5),
            ("wd1_y", 2, 5),
            ("wd1_k", 2, 5),
            ("wd1_l", 2, 5)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'instruments'
    },
    "deep-6": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0),
            ("wd1_k", 0),
            ("wd1_l", 0)
        ],
        "Z": [
            ("y", 2, 6),
            ("k", 2, 6),
            ("l", 2, 6),
            ("wd1_y", 2, 6),
            ("wd1_k", 2, 6),
            ("wd1_l", 2, 6)
        ],
        "options": ["timedumm", "collapse"],
        'category': 'instruments'
    },
    # Different AR lags on y
    "ar1": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 5),
            ("k", 3, 5),
            ("l", 3, 5),
            ("wd1_y", 3, 6),
            ("wd1_k", 3, 6),
            ("wd1_l", 3, 6)
        ],
        "options": ["timedumm", "collapse"]
    },
    # Different AR lags on y
    "ar2": {
        "Y": ("y", 0),
        "X": [
            ("y", 1, 2),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 5),
            ("k", 3, 5),
            ("l", 3, 5),
            ("wd1_y", 3, 6),
            ("wd1_k", 3, 6),
            ("wd1_l", 3, 6)
        ],
        "options": ["timedumm", "collapse"]
    },
    "unknown": {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 4),
            ("k", 3, 4),
            ("l", 3, 4),
            ("wd1_y", 3, 4),
            ("wd1_k", 3, 4),
            ("wd1_l", 3, 4)
        ],
        "options": ["timedumm", "collapse"]
    }
}

from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="model")

outputs = []
category = 'instruments'
categories = dict([(entry.get('category', None), True) for entry in models.values()]).keys()
c_ind = chr(97 + list(categories).index(category))
base_out_name = ("results_3", "dyn")
out_name = f"{base_out_name[0]}{c_ind}_{base_out_name[1]}_{category}_str"

with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
    for m_name, mod in models.items():
        if mod.get("category", '') != category:
            continue
        if mod.get("include", True) is False:
            continue

        y_var = mod["Y"][0]
        x_vars = [unpack_x_var(x) for x in mod["X"]]
        z_vars = [unpack_z_var(z) for z in mod["Z"]]

        combined = [mod["Y"]] + mod["X"] + mod["Z"]
        all_vars = [var[0] for var in combined]
        df_filtered = (
            df_panel_dyn
            .copy()
            .dropna(subset=all_vars)
        )

        command_str = f"{y_var} {' '.join(x_vars)} | {' '.join(z_vars)} | {' '.join(mod['options'])}"
        print(f"Model '{m_name}': generated Command String:")
        print(command_str)
        print("-" * 50)
        
        try:
            sys_gmm_model = regression.abond(command_str, df_filtered, ['registered_number', 'year'])
            outputs.append(sys_gmm_model)
            res = get_dyn_results(sys_gmm_model)
            output_str = get_output_str(res, mod['Z'], m_name)
            f.write(output_str)
            del sys_gmm_model
        except Exception as e:
            print(f"Error occurred while processing model with command: {command_str}")
            print(f"Error details: {e}")

Model 'deep-4': generated Command String:
y L1.y k l wd1_y wd1_k wd1_l | gmm(y, 2:4) gmm(k, 2:4) gmm(l, 2:4) gmm(wd1_y, 2:4) gmm(wd1_k, 2:4) gmm(wd1_l, 2:4) | timedumm collapse
--------------------------------------------------
